# WaveGuard — Colab GPU 자동 라벨링 + yolo26s 파인튜닝

한 노트북으로 **정확 라벨링(교사 모델, GPU)** → **train/val 분할** → **yolo26s 학습** → **가중치 내보내기**를 한다.

## 준비
1. 로컬(vision/)에서: `powershell -ExecutionPolicy Bypass -File .\finetune\pack_for_colab.ps1`
2. 생성된 `vision/finetune/gwangalli_colab.zip` 를 **Google Drive 최상위(MyDrive)** 에 업로드
3. Colab 메뉴 **런타임 > 런타임 유형 변경 > T4 GPU** 선택 후 이 노트북을 위에서부터 실행

> 모델 가중치(yolo26m/yolo26s)는 ultralytics가 자동 다운로드한다. 라벨은 수집 프레임으로부터 GPU가 만든다.
>
> **학습은 `!yolo` CLI가 아니라 Python API(`YOLO(...).train(...)`)를 쓴다.**  
> Colab에서 `yolo: command not found` / `No module named ultralytics.__main__` 가 나는 것을 피하기 위함이다.

## 1. GPU 확인

In [ ]:
!nvidia-smi

## 2. 패키지 설치 (ultralytics + sahi)

In [ ]:
# CLI(yolo / python -m ultralytics)는 Colab에서 PATH·패키지 구조 문제로 자주 깨짐
# → 학습은 반드시 Python API(from ultralytics import YOLO) 사용
!pip -q install -U ultralytics sahi openai-clip
import torch
from ultralytics import YOLO
import ultralytics
ultralytics.checks()
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(CPU — 런타임 유형을 GPU로 바꾸세요)')
print('YOLO OK:', YOLO)

## 3. Google Drive 마운트 + 패키지 압축 해제
업로드한 `gwangalli_colab.zip` 경로를 `PKG` 에 맞게 수정.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, zipfile, shutil
PKG = '/content/drive/MyDrive/gwangalli_colab.zip'   # ← 업로드한 zip 경로
assert os.path.exists(PKG), f'zip 없음: {PKG} (Drive 경로 확인)'
if os.path.exists('/content/pkg'): shutil.rmtree('/content/pkg')
with zipfile.ZipFile(PKG) as z: z.extractall('/content/pkg')
%cd /content/pkg/vision
n = len([f for f in os.listdir('finetune/raw') if f.endswith('.jpg')])
print('raw frames:', n)

## 4. GPU 자동 라벨링 (빠른 경로 — Colab Pro+ 권장)

이전 `--teacher`(SAHI 고배율)는 GPU에서도 장당 수십 초~수분 → 중단되기 쉽다.
대회용은 **yolo26m 단일 추론(`--light`)** 이 훨씬 빠르고, 이후 튜브 라벨(4.2) + 학습으로 보완한다.
결과: `finetune/dataset/{images,labels,preview}/`.


In [ ]:
import os
os.chdir('/content/pkg/vision')
# Colab Pro+: SAHI teacher 대신 LIGHT(yolo26m) — 수백 장도 수분~십수분이면 끝
# 중단 후 재개: --skip-existing
# 더 빠르게: --step 3 / 더 촘촘히: --step 1
!python finetune/prelabel.py --light --model yolo26m.pt --step 2 --skip-existing


## 4.2 튜브 자동 라벨 (class 1) — 타일링(작은 튜브까지)

튜브는 멀리서 전체 프레임 1회 추론으로는 거의 놓친다(이전 tube≈0 원인).
**물 구역을 좌우 타일로 잘라 확대 추론**해야 recall이 오른다.

> 권장: 튜브 라벨은 **로컬 PC에서 미리 생성**해 dataset zip에 포함하면 이 셀을 건너뛰고 6절 학습만 하면 된다.
> Colab GPU에서 다시 라벨하려면 아래 셀을 실행(로컬과 동일한 타일링 스크립트).


In [ ]:
# 튜브 자동 라벨: 물 구역 타일링 YOLO-World (로컬과 동일 로직).
# 이미 로컬에서 라벨해 dataset zip을 올렸다면 이 셀은 건너뛰세요(SKIP).
import os
os.chdir('/content/pkg/vision')
!python finetune/label_tubes_local.py --device 0


In [ ]:
# (선택) 라벨 품질 육안 검수 — preview 몇 장 미리보기
import glob, random
from IPython.display import Image, display
prev = sorted(glob.glob('finetune/dataset/preview/*.jpg'))
print('preview:', len(prev))
for p in random.sample(prev, min(3, len(prev))):
    print(p); display(Image(p, width=720))

## 4.5 의심 프레임 검수 (선택)

자동 라벨 중 '파도 오탐'처럼 이상한 프레임을 골라 `finetune/review/`에 모으고, 아래에서 한 화면에 모두 보여준다. 나쁜 프레임의 파일명을 다음 셀 `bad` 목록에 적어 실행하면 데이터셋에서 제거된다.

**전부 삭제:** 검수 없이 의심을 한 번에 지우려면 `flag_suspect.py --purge` 셀을 실행하세요. (진짜 혼잡 장면도 지워질 수 있음)

In [ ]:
!python finetune/flag_suspect.py
# 의심 프레임을 한 화면에 모두 표시 (박스 그려진 preview)
import glob
from IPython.display import Image, display
sus = sorted(glob.glob('finetune/review/*.jpg'))
print('의심 프레임:', len(sus))
for p in sus:
    print(p)
    display(Image(p, width=640))

In [ ]:
# 의심 프레임 전부 삭제 (검수 생략)
# 한 번 더 확인: 진짜로 붐비는 정상 장면도 삭제될 수 있습니다.
import os
os.chdir('/content/pkg/vision')
!python finetune/flag_suspect.py --purge
print('삭제 후 labels:', len(os.listdir('finetune/dataset/labels')) if os.path.isdir('finetune/dataset/labels') else 0)
print('이제 5절 make_dataset 부터 이어서 실행하세요.')

In [ ]:
# 위에서 본 '나쁜' 프레임 파일명(확장자 없이)을 여기에 적고 실행 → 데이터셋에서 제거
bad = [
    # 'gwangalli_20260730_110821',
    # 'gwangalli_20260730_111037',
]
import os
n = 0
for stem in bad:
    for p in (f'finetune/dataset/images/{stem}.jpg',
              f'finetune/dataset/labels/{stem}.txt',
              f'finetune/dataset/preview/{stem}.jpg',
              f'finetune/review/{stem}.jpg'):
        if os.path.exists(p):
            os.remove(p)
    n += 1
    print('removed', stem)
print(f'제거 {n}장. 이제 아래 5절(make_dataset)부터 이어서 실행하세요.')

## 5. train/val 분할 + data.yaml (절대경로)

In [ ]:
# 작업 폴더가 바뀌어 있어도 복구 후 분할
import os
os.chdir('/content/pkg/vision')
print('cwd:', os.getcwd())
assert os.path.exists('finetune/make_dataset.py'), 'zip 해제 경로 확인: /content/pkg/vision 이 없음'
assert os.path.isdir('finetune/dataset/labels'), '4절 prelabel 먼저 실행하세요 (labels 없음)'
!python finetune/make_dataset.py --val 0.2
# Colab 절대경로로 data.yaml 재작성 (ultralytics 경로 혼선 방지)
root = '/content/pkg/vision/finetune/dataset'
os.makedirs(root, exist_ok=True)
open(root+'/data.yaml','w').write(
    f'path: {root}\ntrain: train/images\nval: val/images\nnames:\n  0: person\n  1: tube\n'
)
print(open(root+'/data.yaml').read())
print('train images:', len(os.listdir(root+'/train/images')) if os.path.isdir(root+'/train/images') else 0)


## 6. yolo26s 파인튜닝 (GPU, Python API)

`!yolo` CLI는 Colab에서 `command not found` / `No module named ultralytics.__main__` 가 자주 납니다.
아래 셀은 `from ultralytics import YOLO` 로 학습합니다.

메모리 부족 시 `batch` 를 4로 낮추거나 `imgsz` 를 768로.

In [ ]:
# 6) 파인튜닝 학습 (Python API — CLI 사용 금지)
import os
os.chdir('/content/pkg/vision')

from ultralytics import YOLO

DATA = '/content/pkg/vision/finetune/dataset/data.yaml'
assert os.path.exists(DATA), f'data.yaml 없음: {DATA} (5절 make_dataset 먼저)'

BASE = 'yolo26s.pt'
try:
    model = YOLO(BASE)
except Exception as e:
    print('yolo26s 로드 실패 → yolov8s로 대체:', e)
    BASE = 'yolov8s.pt'
    model = YOLO(BASE)

print('base model:', BASE)
results = model.train(
    data=DATA,
    epochs=100,
    imgsz=1024,
    batch=8,          # OOM이면 4
    device=0,         # GPU
    patience=30,
    close_mosaic=10,
    hsv_v=0.5,
    degrees=0.0,
    translate=0.05,
    scale=0.3,
    fliplr=0.5,
    project='/content/runs',
    name='gwangalli',
    exist_ok=True,
)
print('best weights:', '/content/runs/gwangalli/weights/best.pt')

## 7. 검증 + 가중치 Drive로 내보내기

In [ ]:
# 7) 검증 (Python API)
from ultralytics import YOLO
import os

BEST = '/content/runs/gwangalli/weights/best.pt'
DATA = '/content/pkg/vision/finetune/dataset/data.yaml'
assert os.path.exists(BEST), f'best.pt 없음: {BEST} (6절 학습 먼저)'

model = YOLO(BEST)
metrics = model.val(data=DATA, imgsz=1024, device=0)
print('mAP50:', round(float(metrics.box.map50), 4))
print('mAP50-95:', round(float(metrics.box.map), 4))
print('precision:', round(float(metrics.box.mp), 4),
      'recall:', round(float(metrics.box.mr), 4))

In [ ]:
import shutil, os
src = '/content/runs/gwangalli/weights/best.pt'
dst = '/content/drive/MyDrive/yolo26s_beach_ft.pt'
shutil.copy(src, dst)
print('saved ->', dst, os.path.getsize(dst)//1024, 'KB')

## 8. 로컬 적용
Drive의 `yolo26s_beach_ft.pt` 를 다운로드해서 다음 위치에 넣기:

```
vision/models/yolo26s_beach_ft.pt
```

`realtime_safety_map.py` 가 `FAST_SAHI_MODEL_CANDIDATES` 첫 항목으로 이 파일을 **최우선 자동 로드**한다. 서버를 재기동하면 파인튜닝된 모델로 탐지한다.